# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

# Each nested line becomes a row; order and vendor_id are copied from the parent.
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
)
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

# A left join preserves every line item and the indicator identifies the gap.
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True,
                     validate='many_to_one')
unmatched = joined.loc[joined['_merge'] == 'left_only']
print('unmatched vendor:', unmatched['vendor_id'].unique().tolist())
print('units attached to unmatched vendor:', unmatched['qty'].sum())
joined

unmatched vendor: ['V-07']
units attached to unmatched vendor: 3


           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Keep the unmatched seller visible instead of losing its three units in groupby.
joined['name'] = joined['name'].fillna('Unknown vendor')
quantity_by_vendor = (joined.groupby('name')['qty']
                      .sum()
                      .sort_values(ascending=False))
quantity_by_vendor

name
Hoos Burgers      3
Unknown vendor    3
Cav Merch         1
Name: qty, dtype: int64

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# Unmatched vendors have no zone; label that gap so all seven units remain visible.
joined['zone'] = joined['zone'].fillna('Unassigned')
quantity_by_zone = (joined.groupby('zone')['qty']
                    .sum()
                    .sort_values(ascending=False))
print(quantity_by_zone)
print('units accounted for:', quantity_by_zone.sum(), 'of 7')

zone
A             4
Unassigned    3
Name: qty, dtype: int64
units accounted for: 7 of 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},
]

# An order with no lines cannot supply a line-item row, so select records that have lines.
records_with_lines = [record for record in ragged_json if 'lines' in record]
ragged_lines = pd.json_normalize(
    records_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id'],
    errors='ignore',
)

# The missing Fries quantity stays NaN, which remains different from a real zero.
print(ragged_lines)
print('missing quantities:', ragged_lines['qty'].isna().sum())

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01
missing quantities: 1


### Q6 — Validate

In [7]:
# Validate the expected shape and confirm the vendor join did not duplicate rows.
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
assert ragged_lines['qty'].isna().sum() == 1
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

_your answer here_